# A 股市场数据日报

**学习目标**：用 AkShare 拉取 A 股指数和个股数据，手算收益率、均线、波动率、最大回撤、Sharpe 比率、成交量、换手率。

**标的清单**：
- 指数：上证指数 (sh000001)
- 个股：贵州茅台 (600519)、五粮液 (000858)、中国平安 (601318)、宁德时代 (300750)、招商银行 (600036)

## 1. 环境准备

In [19]:
import akshare as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date, timedelta
import os

# 中文字体支持
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

# 输出目录
OUTPUT_DIR = "daily_report"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 今天日期
TODAY = date.today().strftime("%Y-%m-%d")
print(f"报告日期: {TODAY}")

报告日期: 2026-05-25


## 2. 数据获取

### 2.1 指数数据（腾讯来源）

`stock_zh_index_daily_tx` 从腾讯财经拉取，数据从 1990 年至今。返回：`['date', 'open', 'close', 'high', 'low', 'amount']`。
腾讯源只提供成交额，没有成交量（手数）。

In [20]:
def fetch_index(symbol, name):
    """拉取指数日线数据，返回标准化的 DataFrame。"""
    df = ak.stock_zh_index_daily_tx(symbol=symbol)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    df = df.rename(columns={
        "date": "日期", "open": "开盘", "close": "收盘",
        "high": "最高", "low": "最低", "amount": "成交额"
    })
    df["涨跌幅"] = df["收盘"].pct_change() * 100
    df["名称"] = name
    df["代码"] = symbol
    return df[["日期", "开盘", "最高", "最低", "收盘", "涨跌幅", "成交额", "名称", "代码"]]


sh = fetch_index("sh000001", "上证指数")
print(f"上证指数: {len(sh)} 行, {sh['日期'].min().date()} ~ {sh['日期'].max().date()}")
sh.tail(3)

  0%|          | 0/37 [00:00<?, ?it/s]

上证指数: 8635 行, 1990-12-19 ~ 2026-05-22


,日期,开盘,最高,最低,收盘,涨跌幅,成交额,名称,代码
8632,2026-05-20,4152.70,4169.85,4139.97,4162.18,-0.176518,624152035.0,上证指数,sh000001
8633,2026-05-21,4174.38,4199.53,4074.22,4077.28,-2.039796,727218701.0,上证指数,sh000001
8634,2026-05-22,4096.17,4120.09,4067.75,4112.90,0.873622,589459644.0,上证指数,sh000001


### 2.2 个股数据（新浪来源）

`stock_zh_a_daily` 返回 `columns = ['date', 'open', 'high', 'low', 'close', 'volume', 'amount', 'outstanding_share', 'turnover']`

**列名映射**：
- `volume` → `成交量`（手，1 手 = 100 股）
- `amount` → `成交额`（元）
- `turnover` → `换手率`（%，当日成交量 / 流通股本）

**参数说明**：
- `symbol='sh600519'`：上海股票加 `sh` 前缀，深圳股票加 `sz` 前缀
- `adjust='qfq'`：前复权，消除分红拆股对价格的影响

In [ ]:
def fetch_stock(symbol, name):
    """拉取个股日线（前复权），返回标准化的 DataFrame。"""
    df = ak.stock_zh_a_daily(symbol=symbol, adjust="qfq")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    df = df.rename(columns={
        "date": "日期", "open": "开盘", "high": "最高",
        "low": "最低", "close": "收盘", "volume": "成交量",
        "amount": "成交额", "turnover": "换手率"
    })
    df["名称"] = name
    df["代码"] = symbol
    df["涨跌幅"] = df["收盘"].pct_change() * 100
    return df[["日期", "开盘", "最高", "最低", "收盘", "涨跌幅",
               "成交量", "成交额", "换手率", "名称", "代码"]]


stocks_config = [
    ("sh600519", "贵州茅台"),
    ("sz000858", "五粮液"),
    ("sh601318", "中国平安"),
    ("sz300750", "宁德时代"),
    ("sh600036", "招商银行"),
]

stocks = {}
for sym, name in stocks_config:
    df = fetch_stock(sym, name)
    stocks[sym] = df
    print(f"{name} ({sym}): {len(df)} 行, {df['日期'].min().date()} ~ {df['日期'].max().date()}")

stocks["sh600519"].tail(3)

### 2.3 取近一年的共同日期窗口

所有标的的日期不完全对齐，统一截取「最近 252 个交易日（约一年）」来比对。

In [ ]:
WINDOW = 252

sh = sh.tail(WINDOW).reset_index(drop=True)
for sym in stocks:
    stocks[sym] = stocks[sym].tail(WINDOW).reset_index(drop=True)

print(f"统一窗口: {WINDOW} 个交易日")
print(f"上证指数: {sh['日期'].min().date()} ~ {sh['日期'].max().date()}")

## 3. 收益率计算

$$
\text{日收益率}_t = \frac{\text{收盘价}_t - \text{收盘价}_{t-1}}{\text{收盘价}_{t-1}}
$$

$$
\text{累计收益率} = \prod_{t=1}^{T}(1 + \text{日收益率}_t) - 1
$$

In [ ]:
def compute_returns(df):
    df = df.copy()
    df["日收益率"] = df["收盘"].pct_change()
    df["累计收益率"] = (1 + df["日收益率"]).cumprod() - 1
    return df


sh = compute_returns(sh)
for sym in stocks:
    stocks[sym] = compute_returns(stocks[sym])

print("上证指数 — 最近 5 天")
print(sh[["日期", "收盘", "日收益率", "累计收益率"]].tail(5).to_string(index=False))
print(f"\n区间累计收益: {sh['累计收益率'].iloc[-1]:.2%}")

### 3.1 收益率对比图

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(sh["日期"], sh["累计收益率"] * 100, linewidth=2, label="上证指数", color="#1f77b4")
for (sym, name), color in zip(stocks_config, ["#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]):
    ax.plot(stocks[sym]["日期"], stocks[sym]["累计收益率"] * 100,
            linewidth=1.2, alpha=0.8, label=name, color=color)
ax.set_title("累计收益率对比 (%)", fontsize=13)
ax.legend(fontsize=7, loc="upper left")
ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.8)
ax.grid(True, alpha=0.3)

ax = axes[1]
heatmap_data = {"上证指数": sh["日收益率"].tail(30).values}
for sym, name in stocks_config:
    heatmap_data[name] = stocks[sym]["日收益率"].tail(30).values
heatmap_df = pd.DataFrame(heatmap_data)
im = ax.imshow(heatmap_df.T * 100, aspect="auto", cmap="RdYlGn", vmin=-5, vmax=5)
ax.set_xticks(range(0, 30, 5))
ax.set_xticklabels(range(1, 31, 5))
ax.set_yticks(range(len(heatmap_df.columns)))
ax.set_yticklabels(heatmap_df.columns, fontsize=8)
ax.set_title("最近 30 日日收益率热力图 (%)", fontsize=13)
fig.colorbar(im, ax=ax, shrink=0.7)

fig.suptitle(f"A 股数据日报 — {TODAY}", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/returns_{TODAY}.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. 移动均线 (MA)

$$
\text{MA}_N(t) = \frac{1}{N} \sum_{i=0}^{N-1} \text{收盘价}_{t-i}
$$

MA5/MA10/MA20/MA60 分别对应一周、两周、一个月、一个季度。

In [ ]:
def compute_ma(df):
    df = df.copy()
    for n in [5, 10, 20, 60]:
        df[f"MA{n}"] = df["收盘"].rolling(n).mean()
    return df


sh = compute_ma(sh)
for sym in stocks:
    stocks[sym] = compute_ma(stocks[sym])

print("上证指数 — 均线数据（最后 5 日）")
print(sh[["日期", "收盘", "MA5", "MA20", "MA60"]].tail(5).to_string(index=False))

In [ ]:
def plot_price_and_ma(df, title, ax):
    ax.plot(df["日期"], df["收盘"], color="black", linewidth=1.5, label="收盘价")
    ax.plot(df["日期"], df["MA5"], linewidth=0.8, alpha=0.8, label="MA5")
    ax.plot(df["日期"], df["MA20"], linewidth=0.8, alpha=0.8, label="MA20")
    ax.plot(df["日期"], df["MA60"], linewidth=0.8, alpha=0.8, label="MA60")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=7, loc="upper left")
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

plot_price_and_ma(sh, "上证指数", axes[0])
for i, (sym, name) in enumerate(stocks_config[:4]):
    plot_price_and_ma(stocks[sym], name, axes[1 + i])

# 最后一格放 MA 状态表
ma_status = []
all_labels = [("上证指数", sh)] + [(n, stocks[s]) for s, n in stocks_config]
for label, df in all_labels:
    last = df.iloc[-1]
    mas = [last.get(f"MA{n}", np.nan) for n in [5, 10, 20, 60]]
    arr = ("多头" if (pd.notna(mas[0]) and pd.notna(mas[3]) and mas[0] > mas[3])
           else ("空头" if (pd.notna(mas[0]) and pd.notna(mas[3]) and mas[0] < mas[3]) else "—"))
    ma_status.append({
        "标的": label, "收盘": f"{last['收盘']:.2f}",
        "MA5": f"{mas[0]:.2f}" if pd.notna(mas[0]) else "—",
        "MA20": f"{mas[2]:.2f}" if pd.notna(mas[2]) else "—",
        "MA60": f"{mas[3]:.2f}" if pd.notna(mas[3]) else "—",
        "多/空": arr,
    })

ma_df = pd.DataFrame(ma_status)
axes[5].axis("off")
tbl = axes[5].table(cellText=ma_df.values, colLabels=ma_df.columns,
                     cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1.2, 1.6)
axes[5].set_title("MA 状态汇总", fontsize=11)

fig.suptitle(f"移动均线 — {TODAY}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ma_{TODAY}.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. 波动率

$$
\sigma_{\text{年}} = \sigma_{\text{日}} \times \sqrt{252}
$$

252 是 A 股一年大约的交易天数。

In [ ]:
def compute_volatility(df):
    df = df.copy()
    df["滚动波动率"] = df["日收益率"].rolling(20).std()
    df["年化波动率"] = df["滚动波动率"] * np.sqrt(252) * 100
    return df


sh = compute_volatility(sh)
for sym in stocks:
    stocks[sym] = compute_volatility(stocks[sym])

print("=== 年化波动率（最新） ===")
all_labels = [("上证指数", sh)] + [(n, stocks[s]) for s, n in stocks_config]
for label, df in all_labels:
    print(f"  {label}: {df['年化波动率'].iloc[-1]:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
plot_items = [("上证指数", sh, "#1f77b4")] + \
             [(name, stocks[sym], c) for (sym, name), c in
              zip(stocks_config[:3], ["#ff7f0e", "#2ca02c", "#d62728"])]
for label, df, color in plot_items:
    ax.plot(df["日期"], df["年化波动率"], linewidth=1.2, label=label, color=color)
ax.set_title(f"20 日滚动年化波动率 — {TODAY}", fontsize=13)
ax.set_ylabel("年化波动率 (%)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/volatility_{TODAY}.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. 最大回撤 (Max Drawdown)

$$
\text{Drawdown}_t = \frac{\text{净值}_t - \text{历史最高净值}_t}{\text{历史最高净值}_t}
$$

$$
\text{Max Drawdown} = \min(\text{Drawdown}_1, \dots, \text{Drawdown}_T)
$$

In [ ]:
def compute_drawdown(df):
    df = df.copy()
    df["净值"] = (1 + df["日收益率"]).cumprod()
    df["历史最高"] = df["净值"].cummax()
    df["回撤"] = (df["净值"] - df["历史最高"]) / df["历史最高"]
    return df


sh = compute_drawdown(sh)
for sym in stocks:
    stocks[sym] = compute_drawdown(stocks[sym])

print("=== 最大回撤 ===")
all_labels = [("上证指数", sh)] + [(n, stocks[s]) for s, n in stocks_config]
for label, df in all_labels:
    mdd = df["回撤"].min() * 100
    mdd_date = df.loc[df["回撤"].idxmin(), "日期"].date()
    print(f"  {label}: {mdd:.1f}%  (日期: {mdd_date})")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
plot_items = [("上证指数", sh, "#1f77b4")] + \
             [(name, stocks[sym], c) for (sym, name), c in
              zip(stocks_config[:3], ["#ff7f0e", "#2ca02c", "#d62728"])]
for label, df, color in plot_items:
    ax.fill_between(df["日期"], df["回撤"] * 100, 0, alpha=0.3, label=label, color=color)
ax.set_title(f"回撤曲线 — {TODAY}", fontsize=13)
ax.set_ylabel("回撤 (%)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color="gray", linewidth=0.5)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/drawdown_{TODAY}.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. 成交量与换手率

### 成交量

成交量 = 当日成交的股票手数（1 手 = 100 股）。成交量是**市场热度**的温度计：
- **放量上涨**：量价齐升，上涨有资金支撑 → 趋势确认
- **缩量上涨**：价涨量不涨，追高意愿不足 → 可能回调
- **放量下跌**：恐慌抛售，有人在接盘也有人在跑 → 短期可能有分歧

### 换手率

$$
\text{换手率} = \frac{\text{当日成交量}}{\text{流通股本}} \times 100\%
$$

- 换手率是成交量相对于流通股的比率，**消除了股本大小的影响**，不同股票之间可以直接比较
- **< 1%**：冷门，交投清淡
- **1% ~ 3%**：正常活跃度
- **3% ~ 10%**：高度活跃，资金关注
- **> 10%**：极度活跃，短线资金博弈，波动大

In [ ]:
def compute_volume_ma(df):
    """计算成交量和换手率的移动均值。"""
    df = df.copy()
    df["成交量_MA5"] = df["成交量"].rolling(5).mean()
    df["成交量_MA20"] = df["成交量"].rolling(20).mean()
    df["换手率_MA5"] = df["换手率"].rolling(5).mean()
    return df


for sym in stocks:
    stocks[sym] = compute_volume_ma(stocks[sym])

# 打印最新数据
vol_cols = ["日期", "收盘", "成交量", "成交量_MA5", "成交量_MA20", "换手率", "换手率_MA5"]
print("贵州茅台 — 成交量与换手率（最后 5 日）")
df = stocks["sh600519"]
display_cols = ["日期", "收盘", "成交量", "成交量_MA5", "换手率", "换手率_MA5"]
print(df[display_cols].tail(5).to_string(index=False))

### 7.1 成交量柱状图（最近 60 日）

柱子 = 每日成交量，红线 = 20 日均量。**高于红线 = 放量，低于红线 = 缩量。**

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (sym, name) in enumerate(stocks_config):
    ax = axes[i]
    df = stocks[sym].tail(60)
    bars = ax.bar(df["日期"], df["成交量"] / 1e4, color="steelblue", alpha=0.7, width=0.8)
    ax.plot(df["日期"], df["成交量_MA20"] / 1e4, color="red", linewidth=1.2, label="20日均量")
    ax.set_title(name, fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, axis="y")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)

# 最后一格放换手率对比
ax = axes[5]
for (sym, name), color in zip(stocks_config, ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]):
    df = stocks[sym].tail(60)
    ax.plot(df["日期"], df["换手率"], linewidth=1.2, label=name, color=color, alpha=0.8)
ax.set_title("换手率对比 (%)", fontsize=10)
ax.legend(fontsize=6, loc="upper left")
ax.grid(True, alpha=0.3)
ax.tick_params(axis="x", rotation=45, labelsize=7)
ax.tick_params(axis="y", labelsize=7)

fig.suptitle(f"成交量与换手率 — {TODAY}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/volume_{TODAY}.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 量价关系：近期放/缩量情况

当天成交量 > 20 日均量 × 1.2 → **放量**；< 0.8 → **缩量**。结合涨跌判断信号。

In [ ]:
def volume_signal(df):
    """判断最近一天的成交量信号。"""
    last = df.iloc[-1]
    vol_ratio = last["成交量"] / last["成交量_MA20"] if last["成交量_MA20"] > 0 else 1.0
    if vol_ratio > 1.2:
        vol_label = "放量"
    elif vol_ratio < 0.8:
        vol_label = "缩量"
    else:
        vol_label = "正常"

    chg = last["涨跌幅"]
    if vol_label == "放量" and chg > 1:
        signal = "放量上涨 ✓"
    elif vol_label == "放量" and chg < -1:
        signal = "放量下跌 ✗"
    elif vol_label == "缩量" and chg > 1:
        signal = "缩量上涨 △"
    elif vol_label == "缩量" and chg < -1:
        signal = "缩量下跌 —"
    else:
        signal = vol_label
    return signal, vol_ratio


print(f"{'标的':<8} {'换手率':<10} {'量比':<8} {'信号'}")
print("-" * 45)
for sym, name in stocks_config:
    df = stocks[sym]
    last = df.iloc[-1]
    signal, vol_ratio = volume_signal(df)
    turnover = last["换手率"]
    print(f"{name:<6} {turnover:<8.2f}% {vol_ratio:<8.2f} {signal}")

## 8. Sharpe 比率

$$
\text{Sharpe} = \frac{\text{年化收益率} - r_f}{\text{年化波动率}}
$$

- 年化收益率 = 日均收益率 × 252
- $r_f$ = 无风险利率，这里取 2%
- 年化波动率 = 日收益率标准差 × √252

In [ ]:
def compute_sharpe(df, rf=0.02):
    """根据日收益率序列计算年化 Sharpe 比率。"""
    returns = df["日收益率"].dropna()
    daily_mean = returns.mean()
    daily_std = returns.std()
    annual_return = daily_mean * 252
    annual_vol = daily_std * np.sqrt(252)
    sharpe = (annual_return - rf) / annual_vol
    return {"年化收益率": annual_return, "年化波动率": annual_vol, "Sharpe": sharpe}


sharpe_data = []
all_labels = [("上证指数", sh)] + [(n, stocks[s]) for s, n in stocks_config]
for label, df in all_labels:
    s = compute_sharpe(df)
    sharpe_data.append({
        "标的": label,
        "年化收益率": f"{s['年化收益率']:.2%}",
        "年化波动率": f"{s['年化波动率']:.2%}",
        "Sharpe": f"{s['Sharpe']:.2f}",
    })

sharpe_df = pd.DataFrame(sharpe_data)
print(sharpe_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
names = [d["标的"] for d in sharpe_data]
sharpes = [float(d["Sharpe"]) for d in sharpe_data]
colors = ["#2ca02c" if s > 0 else "#d62728" for s in sharpes]

bars = ax.barh(names, sharpes, color=colors, edgecolor="white")
ax.axvline(x=0, color="gray", linewidth=0.8)
ax.set_xlabel("Sharpe 比率", fontsize=12)
ax.set_title(f"年化 Sharpe 比率对比 — {TODAY}", fontsize=13)
ax.grid(True, alpha=0.3, axis="x")
for bar, val in zip(bars, sharpes):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sharpe_{TODAY}.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. 汇总表 & 保存

In [ ]:
summary_rows = []
all_labels = [("上证指数", sh)] + [(n, stocks[s]) for s, n in stocks_config]
for label, df in all_labels:
    last = df.iloc[-1]
    cum_ret = df["累计收益率"].iloc[-1]
    mdd = df["回撤"].min()
    s = compute_sharpe(df)
    signal, _ = volume_signal(df) if "成交量" in df.columns else ("—", 1.0)
    row = {
        "标的": label,
        "最新收盘": round(last["收盘"], 2),
        "日涨跌幅(%)": round(last["涨跌幅"], 2) if pd.notna(last["涨跌幅"]) else "—",
        "MA5": round(last["MA5"], 2) if pd.notna(last.get("MA5", np.nan)) else "—",
        "MA20": round(last["MA20"], 2) if pd.notna(last.get("MA20", np.nan)) else "—",
        "MA60": round(last["MA60"], 2) if pd.notna(last.get("MA60", np.nan)) else "—",
        "换手率(%)": f"{last['换手率']:.2f}" if "换手率" in df.columns and pd.notna(last.get("换手率", np.nan)) else "—",
        "量价信号": signal,
        "区间累计收益": f"{cum_ret:.2%}",
        "最大回撤": f"{mdd:.2%}",
        "年化波动率": f"{s['年化波动率']:.2%}",
        "Sharpe": round(s["Sharpe"], 2),
        "报告日期": TODAY,
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

summary_df.to_csv(f"{OUTPUT_DIR}/market_report_{TODAY}.csv", index=False, encoding="utf-8-sig")
print(f"\n报告已保存至: {OUTPUT_DIR}/market_report_{TODAY}.csv")

## 10. 知识点回顾

| 概念 | 公式 | pandas 函数 |
|------|------|-----------|
| **日收益率** | (今收 - 昨收) / 昨收 | `pct_change()` |
| **累计收益率** | ∏(1+日收益) - 1 | `cumprod()` |
| **移动均线** | N 日收盘均值 | `rolling(N).mean()` |
| **年化波动率** | σ_日 × √252 | `rolling().std()` × √252 |
| **最大回撤** | min((净值-历史最高)/历史最高) | `.cummax()` |
| **成交量均线** | N 日成交量均值 | `rolling(N).mean()` |
| **换手率** | 成交量 / 流通股本 × 100% | 新浪直接提供 |
| **量比** | 当日成交量 / 20日均量 | 手算 |
| **Sharpe** | (年化收益 - 2%) / 年化波动 | 手算 |

**数据源**：指数用腾讯 `stock_zh_index_daily_tx`，个股用新浪 `stock_zh_a_daily`。

**下一步**：改成 .py 脚本用 Windows 任务计划每日自动跑；加入 MACD/RSI 等技术指标。